In [1]:
import pickle
import numpy as np
import gensim
from spec2vec import SpectrumDocument, SpectrumDocumentWithLosses, Spec2Vec

# Include neutral-loss words ("loss@18.01") alongside peak words, as in the original
# Spec2Vec paper. The pretrained model was trained with losses from 5 to 200 Da.
# spec2vec >= 0.9 only adds losses through SpectrumDocumentWithLosses (opt-in).
USE_LOSSES = True
LOSS_MZ_FROM, LOSS_MZ_TO = 5.0, 200.0
SUFFIX = "" if USE_LOSSES else "_peaks_only"

## Load the Cleaned Spectra

Load the cleaned, structure-validated spectra that `preprocess.ipynb` saved as pickle files, one list for pesticides and one for pharmaceuticals. They are in the same order as the rows and columns of the Tanimoto ground-truth matrices, so the Spec2Vec matrices built here line up with them index for index.

In [2]:
# Load the cleaned spectra data for pesticides and pharmaceuticals
with open("/Users/timvandriel/Desktop/CSCI558/project/data/pesticides_clean_spectra.pkl", "rb") as f:
    pest_spectra = pickle.load(f)
    
with open("/Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_clean_spectra.pkl", "rb") as f:
    pharma_spectra = pickle.load(f)

## Load the Spec2Vec Model and Tokenize the Spectra

Spec2Vec treats a mass spectrum like a text document. Each peak becomes a "word" (for example `peak@238.11`), and a Word2Vec model learns an embedding for each word from how peaks co-occur across many spectra.

1. **Load the model**: load the pretrained Word2Vec model (`spec2vec_positive.model`, trained on positive-ion-mode spectra) and print its vocabulary size.
2. **Tokenize**: turn every spectrum into a document of words rounded to 2 decimal places.
   - With `USE_LOSSES = True` (default), use `SpectrumDocumentWithLosses`: each peak gives a `peak@` word, and each neutral loss (precursor m/z − peak m/z) between 5 and 200 Da gives a `loss@` word. This matches how the pretrained model was trained and how the original paper scored spectra. Losses use the same intensity as their peak as their weight.
   - With `USE_LOSSES = False`, use `SpectrumDocument` (peak words only). This is the default behavior of spec2vec ≥ 0.9, which ignores the model's 19,501 `loss@` vectors.
3. **Check vocabulary coverage**: for the first pesticide spectrum, count how many of its words exist in the model's vocabulary. Words the model has never seen get no embedding, so high coverage means the model can represent these spectra well.

In [3]:
# 1. Load weights
model = gensim.models.Word2Vec.load("/Users/timvandriel/Desktop/CSCI558/project/models/spec2vec_positive.model")
print(f"Model loaded. Vocabulary size: {len(model.wv.key_to_index):,} tokens.")

# 2. Tokenize cleaned spectra (peaks + neutral losses, or peaks only)
def make_document(spectrum):
    if USE_LOSSES:
        return SpectrumDocumentWithLosses(spectrum, n_decimals=2,
                                          loss_mz_from=LOSS_MZ_FROM, loss_mz_to=LOSS_MZ_TO)
    return SpectrumDocument(spectrum, n_decimals=2)

pest_docs = [make_document(s) for s in pest_spectra]
pharma_docs = [make_document(s) for s in pharma_spectra]

# 3. Check vocabulary coverage on document 0
doc0_words = pest_docs[0].words
known = [w for w in doc0_words if w in model.wv.key_to_index]
n_peak = sum(w.startswith("peak@") for w in doc0_words)
n_loss = sum(w.startswith("loss@") for w in doc0_words)
print(f"Doc 0 Tokens: {len(doc0_words)} total ({n_peak} peaks, {n_loss} losses) | "
      f"{len(known)} in vocabulary ({len(known)/len(doc0_words)*100:.1f}%)")

Model loaded. Vocabulary size: 115,910 tokens.


Doc 0 Tokens: 58 total (30 peaks, 28 losses) | 58 in vocabulary (100.0%)


## Define the Spec2Vec Similarity Function

Set up the `Spec2Vec` similarity scorer. Each spectrum's embedding is a weighted sum of its peak-word vectors, and the similarity between two spectra is the cosine similarity of their embeddings.
- `intensity_weighting_power=0.5`: each peak is weighted by the square root of its intensity, so strong peaks count more without completely dominating.
- `allowed_missing_percentage=15.0`: a spectrum is still scored if up to 15% of its (weighted) peaks are missing from the model's vocabulary.

`compute_similarity_matrix` then:
1. Computes the all-vs-all n × n Spec2Vec similarity matrix and casts it to `float32`.
2. Asserts that the diagonal is ~1.0 and the matrix is symmetric (within a 1e-3 tolerance).
3. Prints the mean, median, and max of the off-diagonal (upper-triangle) scores.

This cell only defines the function. The next cell runs it.

In [4]:
spec2vec_similarity = Spec2Vec(model=model, intensity_weighting_power=0.5, allowed_missing_percentage=15.0)

def compute_similarity_matrix(docs):
    print(f"Computing similarity matrix for {len(docs)} documents...")
    # compute similarity matrix
    sim_matrix = spec2vec_similarity.matrix(docs, docs)
    
    # Cast to float32
    sim_matrix = sim_matrix.astype(np.float32)
    
    # Sanity checks
    assert np.allclose(np.diag(sim_matrix), 1.0, atol=1e-3), "Diagonal self-similarity should be ~1.0"
    assert np.allclose(sim_matrix, sim_matrix.T, atol=1e-3), "Spec2Vec matrix should be symmetric"
    
    # Distribution summary of off-diagonal pairs
    off_diag = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    print(f"Off-diagonal Spec2Vec -> Mean: {off_diag.mean():.4f}, Median: {np.median(off_diag):.4f}, Max: {off_diag.max():.4f}")
    
    return sim_matrix
    
    

## Compute and Save the Spec2Vec Matrices

Run `compute_similarity_matrix` on both datasets: 653 × 653 for pesticides and 377 × 377 for pharmaceuticals. Save each result to `project/data/` as a `.npy` file so later notebooks can load the matrices without recomputing them:
- `USE_LOSSES = True`: `pest_spec2vec_matrix.npy` and `pharma_spec2vec_matrix.npy` (peaks + losses, the main result)
- `USE_LOSSES = False`: `pest_spec2vec_matrix_peaks_only.npy` and `pharma_spec2vec_matrix_peaks_only.npy`

In [5]:
pest_sim_matrix = compute_similarity_matrix(pest_docs)
pharma_sim_matrix = compute_similarity_matrix(pharma_docs)

# Save the computed spectral similarity matrices
for name, mat in [("pest", pest_sim_matrix), ("pharma", pharma_sim_matrix)]:
    out_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/{name}_spec2vec_matrix{SUFFIX}.npy"
    np.save(out_file, mat)
    print(f"Saved: {out_file} (shape: {mat.shape})")

Computing similarity matrix for 653 documents...


Off-diagonal Spec2Vec -> Mean: 0.0666, Median: 0.0543, Max: 0.9982
Computing similarity matrix for 377 documents...


Off-diagonal Spec2Vec -> Mean: 0.1264, Median: 0.1208, Max: 0.9657
Saved: /Users/timvandriel/Desktop/CSCI558/project/data/pest_spec2vec_matrix.npy (shape: (653, 653))
Saved: /Users/timvandriel/Desktop/CSCI558/project/data/pharma_spec2vec_matrix.npy (shape: (377, 377))


## Compare Spec2Vec with Structural Similarity

Load the Tanimoto ground-truth matrices from `preprocess.ipynb` and measure how well Spec2Vec scores track true structural similarity.

`evaluate_alignment` takes the upper triangle of both matrices, which gives every unique pair of spectra once and excludes self-comparisons. It then computes two correlations:
- **Pearson (r)**: how linearly the Spec2Vec scores follow the Tanimoto scores.
- **Spearman (ρ)**: whether the two scores rank the pairs in the same order, whether or not the relationship is linear.

Higher values mean spectral similarity is a better proxy for structural similarity.

In [6]:
from scipy.stats import spearmanr, pearsonr
pest_matrix = np.load("/Users/timvandriel/Desktop/CSCI558/project/data/pesticides_tanimoto_matrix.npy")
pharma_matrix = np.load("/Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_tanimoto_matrix.npy")
def evaluate_alignment(ground_truth_mat, pred_mat, name="Pesticides"):
    # Extract upper triangle excluding self-similarity diagonal
    triu_idx = np.triu_indices_from(ground_truth_mat, k=1)
    y_true = ground_truth_mat[triu_idx]
    y_pred = pred_mat[triu_idx]
    
    pearson_corr, _ = pearsonr(y_true, y_pred)
    spearman_corr, _ = spearmanr(y_true, y_pred)
    
    print(f"\n--- {name}: Spec2Vec vs. Structural Tanimoto ---")
    print(f"Total evaluated pairs: {len(y_true):,}")
    print(f"Pearson Correlation (r):  {pearson_corr:.4f}")
    print(f"Spearman Correlation (ρ): {spearman_corr:.4f}")

evaluate_alignment(pest_matrix, pest_sim_matrix, "Pesticides")
evaluate_alignment(pharma_matrix, pharma_sim_matrix, "Pharmaceuticals")


--- Pesticides: Spec2Vec vs. Structural Tanimoto ---
Total evaluated pairs: 212,878
Pearson Correlation (r):  0.4179
Spearman Correlation (ρ): 0.1451

--- Pharmaceuticals: Spec2Vec vs. Structural Tanimoto ---
Total evaluated pairs: 70,876
Pearson Correlation (r):  0.2439
Spearman Correlation (ρ): 0.1723


## Spec2Vec Scores by Structural Similarity Bin

A single correlation can hide what is happening at different similarity levels, especially when most pairs are structurally unrelated. `stratify_scores` groups every unique pair into bins by its true Tanimoto similarity:

| Tanimoto range | Label |
|---|---|
| 0.0–0.2 | Unrelated |
| 0.2–0.4 | Distant |
| 0.4–0.6 | Moderate |
| 0.6–0.8 | Close |
| 0.8–1.0 | Identical/Isomer |

For each bin it reports the number of pairs and the mean, median, and standard deviation of the Spec2Vec scores. If Spec2Vec captures structure well, its scores should rise steadily from the Unrelated bin to the Identical/Isomer bin.

In [7]:
import numpy as np
import pandas as pd

def stratify_scores(ground_truth_mat, pred_mat, name="Pesticides"):
    triu_idx = np.triu_indices_from(ground_truth_mat, k=1)
    y_true = ground_truth_mat[triu_idx]
    y_pred = pred_mat[triu_idx]
    
    # Define Tanimoto similarity bins
    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.0-0.2 (Unrelated)", "0.2-0.4 (Distant)", "0.4-0.6 (Moderate)", "0.6-0.8 (Close)", "0.8-1.0 (Identical/Isomer)"]
    
    bin_assignments = pd.cut(y_true, bins=bins, labels=labels, include_lowest=True)
    df = pd.DataFrame({"true_tanimoto": y_true, "spec2vec": y_pred, "bin": bin_assignments})
    
    summary = df.groupby("bin", observed=False)["spec2vec"].agg(
        Pair_Count="count",
        Mean_Spec2Vec="mean",
        Median_Spec2Vec="median",
        Std_Spec2Vec="std"
    ).reset_index()
    
    print(f"\n{'='*55}")
    print(f"Spec2Vec Scores by Structural Similarity Bin: {name.upper()}")
    print(f"{'='*55}")
    print(summary.to_string(index=False))
    return summary

pest_strat = stratify_scores(pest_matrix, pest_sim_matrix, "Pesticides")
pharma_strat = stratify_scores(pharma_matrix, pharma_sim_matrix, "Pharmaceuticals")


Spec2Vec Scores by Structural Similarity Bin: PESTICIDES
                       bin  Pair_Count  Mean_Spec2Vec  Median_Spec2Vec  Std_Spec2Vec
       0.0-0.2 (Unrelated)      193301       0.058430         0.051888      0.078645
         0.2-0.4 (Distant)       16193       0.106635         0.074106      0.139436
        0.4-0.6 (Moderate)        1636       0.210541         0.150483      0.203401
           0.6-0.8 (Close)         520       0.333258         0.292317      0.225110
0.8-1.0 (Identical/Isomer)        1228       0.528199         0.574087      0.347177

Spec2Vec Scores by Structural Similarity Bin: PHARMACEUTICALS
                       bin  Pair_Count  Mean_Spec2Vec  Median_Spec2Vec  Std_Spec2Vec
       0.0-0.2 (Unrelated)       68744       0.123470         0.119014      0.106608
         0.2-0.4 (Distant)        1889       0.196121         0.175079      0.128932
        0.4-0.6 (Moderate)         130       0.357255         0.356461      0.163206
           0.6-0.8 (Close)   